**F검정**
- 사례) 한 연구자는 두 가지 교육 방법(A, B)이 학생들의 성적 분산에 차이가 있는지 알아보고자 한다. 각 집단에서 무작위로 10명씩 표본을 추출하여 시험 점수를 기록하였다.

In [2]:
import pandas as pd

df = pd.DataFrame({
    "A": [85, 88, 90, 78, 85, 86, 91, 89, 84, 87,
          88, 82, 85, 90, 86, 84, 87, 83, 89, 85],
    "B": [80, 76, 92, 85, 79, 94, 88, 90, 95, 77,
          91, 85, 96, 82, 97, 75, 93, 84, 90, 78]
})

# F 검정: 두 집단의 분산 비교... 직접 계산(분산비)
# 분산분석(ANOVA): 세 집단 이상 평균 비교...f_oneway()

1. 통계적 가설 설정
 - 귀무가설(H0) : 두 집단의 모분산은 같다.(𝝈_𝑨^𝟐=𝝈_𝑩^𝟐)
 - 대립가설(H1) : 두 집단의 모분산은 다르다. (𝝈_𝑨^𝟐≠𝝈_𝑩^𝟐)
2. 유의수준 결정
 - 유의수준 𝜶=𝟎.𝟎𝟓

3. 검정통계량
 - F검정 통계량 공식 :  𝑭=(𝑺_큰분산^𝟐)/(𝑺_작은분산^𝟐 )
   (여기서 S^2는 표본분산)
  

In [5]:
var_A = df['A'].var() # A의 분산
var_B = df['B'].var() # B의 분산

if var_A > var_B: # 큰 걸 작은 걸로 나누기
    f_stat = var_A / var_B
else:
    f_stat = var_B / var_A

print("F-statistic:", f_stat)

F-statistic: 5.288213132400431


**판다스의 분산과 넘파이의 분산**

In [ ]:
# 분산: 데이터가 평균에서 흩어진 정도
# 표본분산은 분모가 n-1

import numpy as np
print('Pandas Var :', df['A'].var(), df['B'].var())

# ddof=1: 1을 빼주겠다는 표시를 해줘야 표본분산 계산 가능
print('Numpy Var : ', np.var(df['A'], ddof=1), np.var(df['B'], ddof=1))

Pandas Var : 9.77894736842105 51.71315789473684
Numpy Var :  9.77894736842105 51.71315789473684


**cdf를 통한 p-value 계산**

In [ ]:
# cdf(누적분포함수): 확률 변수가 특정 값보다 작거나 같을 확률을 나타내는 함수

# 단측 검정: (좌측)p = CDF(F, df1, df2), (우측)p = 1 - CDF(F, df1, df2)
# 양측 검정: p = 2 * min(CDF(F, df1, df2), 1 - CDF(F, df1, df2))
# 왼쪽 꼬리는 CDF, 오른쪽 꼬리는 1 - CDF... 둘 중 작은 값을 2배

from scipy import stats

if var_A > var_B:
  df1 = len(df['A']) - 1
  df2 = len(df['B']) - 1
else:
  df1 = len(df['B']) - 1
  df2 = len(df['A']) - 1

p_value = 2 * min(1 - stats.f.cdf(f_stat, df1, df2),
                  stats.f.cdf(f_stat, df1, df2))
print(p_value)


# 바틀렛 검정: 2개 이상 집단의 분산 비교, 정규성 확신, 이상치 x, 카이제곱 분포... bartlett
# 레빈 검정: 2개 이상 집단의 분산 비교, 정규확/이상치 불확실, F 분포...levene

0.0006617791910468185




---


**일원분산분석(One-way ANOVA)**
- 사례) 한 카페 체인에서는 3가지 원두 종류(브라질, 에티오피아, 콜롬비아)를 사용했을 때, 커피의 카페인 함량이 달라지는지 알아보려 한다. 각 원두로 커피를 8잔씩 추출해 카페인 함량(단위: mg)을 측정하였다.

1. 통계적 가설 설정
 - 귀무가설(H0) : 세 원두의 평균 카페인 함량은 같다. (𝛍_𝑨=𝛍_𝑩=𝛍_𝑪)
 - 대립가설(H1) : 적어도 한 원두의 평균 카페인 함량이 다르다. (𝛍_𝑨≠𝛍_𝑩≠𝛍_𝑪)
 - 독립성(기본 가정), 정규성(Shapiro-Wilk), 등분산성(Levene) 가정
2. 유의수준 결정
 - 유의수준 𝜶=𝟎.𝟎𝟓

In [8]:
import pandas as pd

df = pd.DataFrame({
    "Brazil": [95, 97, 94, 100, 96, 98, 99, 97],
    "Ethiopia": [88, 90, 92, 89, 91, 90, 93, 89],
    "Colombia": [99, 101, 98, 100, 102, 97, 99, 101]
})

3. 검정방법
 - 분산분석(ANOVA)은 집단 간 평균 차이를 F-통계량으로 평가

In [9]:
from scipy import stats

f_stat, p_value = stats.f_oneway(df['Brazil'], df['Ethiopia'], df['Colombia'])

print("F-statistic:", f_stat)
print("p-value:", p_value)

F-statistic: 58.32467532467532
p-value: 2.6679263806678924e-09


4. 결과도출
 - F-statistic과 p-value를 확인하고, p-value < 0.05이면 귀무가설 기각
 - 해석 : 유의수준 0.05에서 p-value가 2.66e-09이므로, 귀무가설을 기각 → 세 원두의 평균 카페인 함량이 모두 같다고 보기 어렵다.
